# SQL-04

## Prepare the environment

---

In [ ]:
pip install ipython-sql psycopg2

In [ ]:
%load_ext sql

## Data Definition Lanaguage (DDL)
The SQL data-definition language (DDL) allows specification of:
* Schema for each relation
* Data type of values associated with each attribute
* Integrity constraints
* Creating, deleting, and updating tables
* Views 
* Constraints


First create a new database from command line

postgres# `create database cricdb;`

In [ ]:
%sql postgresql://postgres@localhost:5432/cricdb

---
## Data types in SQL
---

### Basic Types
* **char($n$) or character($n$)**. Fixed length character string, with user specified length $n$.  
* **varchar($n$) or varying character($n$)**. Variable length character strings, with user-specified maximum length $n$.
* **int or integer**. Integer (a finite subset of the integers that is machine-dependent)
* **smallint**. Small integer (a machine-dependent subset of the integer domain type).
* **numeric($m,n$)** $m \implies$ number of digits, and $n \leq m \implies$ number of digits that are to the right of the decimal point (ex. numeric(3,1) allows 44.5 to be stored exactly, but not 0.32 or 444.5).
* **float($n$)**. floating point number of user-specified $n$ digits
* real, double precision

Note: Each type may include a **null** value


### Other data types
* date
* time
* timestamp
* clob
* blob
* user-defined types

---

## Creating/Updating/Deletions
```
create table table_name (
    attribute_1 data_type_1,
    attribute_2 data_type_2,
    ...
    attribute_n data_type_n
)
```

In [ ]:
%sql drop table if exists player

In [ ]:
%%sql create table player (
    player_id integer,
    first_name varchar(25),
    last_name varchar(25),
    dob date,
    team_id integer
)

In [ ]:
%sql select * from pg_catalog.pg_tables limit 10;

In [ ]:
%%sql create table team (
    team_id integer,
    team_name varchar(50),
    team_owner varchar(50)
)

Lets insert some values into the two relations

In [ ]:
%%sql insert into player values
    (1, 'Hardik', 'Pandya', '1993-10-11', 101),
    (2, 'MS', 'Dhoni', '1981-07-07', 102),
    (3, 'Rohit', 'Sharma', '1987-04-30', 101),
    (4, 'Ruturaj', 'Gaikwad', '1997-01-31', 102);


In [ ]:
%%sql insert into team values
    (101, 'Mumbai Indians', 'Reliance Industry Ltd'),
    (102, 'Chennai Super Kings', 'India Cements Ltd'),
    (103, 'Delhi Capitals', 'JSW Sports')
    ;

In [ ]:
%sql select * from player

### Deletions

```
delete from table_name
    where condition;
```

Detele all players from team `102`

In [ ]:
%%sql delete from player
where team_id = 102;

In [ ]:
%sql select * from player

In [ ]:
%%sql delete from player
where team_id IN (
    select team_id from team where team_name = 'Delhi Capitals'
);

### Updating tables
* Change value in a tuple

In [ ]:
%sql select * from player;

In [ ]:
%%sql update player
set team_id = 102 
where player.first_name = 'Hardik' and player.last_name = 'Pandya'; 

In [ ]:
%sql select * from player;

### Altering tables

```
alter table table_name add attribute data_type
alter table table_name drop attribute
delete from table_name
drop table table_name
```

Add a new attribute `position` to the relation `player`

In [ ]:
%sql alter table player add position smallint;

In [ ]:
%sql select * from player;

Q: Update the `position` values in the `player` relation.

In [ ]:
%sql update player set position = 4 where player_id = 1;
%sql update player set position = 1 where player_id = 3;
%sql update player set position = 3 where player_id = 2;

In [ ]:
%sql select * from player

In [ ]:
%sql alter table player drop position;

## Views

* Sometimes, we may want to hide the entire logical model from certain users.

* A view provides a mechanism to hide certain data from the view of certain users

* Any relation that is not of the conceptual model but is made visible to a user as a "virtual relation" is called a view

* Two type of views
    1. Virtual views
    2. Materialized views

* To create a view

    **create view** view_name **as** \<query_expression\>

Create a view `top_mi_csk_players` (top order batters from 'Mumbai Indians' and 'Chennai Super Kings')

In [ ]:
%%sql create view top_mi_csk_players as
select first_name, last_name, team_name 
from player join team on player.team_id = team.team_id
where team_name = 'Mumbai Indians' or team_name = 'Chennai Super Kings'
and position < 5;

In [ ]:
%sql select * from top_mi_csk_players;

To create a materialized view,
``` create materialized view as <query>```

* Q. What is the difference between a view and the CTE defined in `WITH` clause?

---
## Integrity Constraints
* Crucial for data consistency
* Usually a part of schema design and is specified with `create table` statement

In [ ]:
%sql insert into player values (1, 'Hardik', 'Pandya', null, null);

In [ ]:
%sql select * from player;

In [ ]:
%sql drop view top_mi_csk_players;
%sql drop table player;


### Common constraints
* not null
* primary key
* unique
* check(*condition*)
* referential-integrity

In [ ]:
%%sql create table player (
    player_id integer not null,
    first_name varchar(25) not null,
    last_name varchar(25) not null,
    dob date not null,
    team_id integer,
    primary key (player_id),
    unique(player_id, dob),
    check (dob > '1990-01-31'),
    foreign key (team_id) references team (team_id)
)

In [ ]:
%sql drop table if exists team


In [ ]:

%%sql create table team (
    team_id integer,
    team_name varchar(50) not null,
    team_owner varchar(50) not null,
    primary key (team_id)
)

---

## Exercises

* Q: Create or update the database `cricdb`, by adding more relations. Take the ERD discussed in class as an example.

* Q. Try different DDL statements

* Q. Try update statements on view. What will happen when you update the view `top_mi_csk_players` as ('5', 'Ravindra', 'Jadeja')? Will it update the "base" tables?

* Q. Try updating schema of an existing relation by adding a constraint? what happens if the existing values don't satisfy the constraints?

* Q. Write some insert statements to violate different types of constraints.

* Q. What happens when you delete a table, which has been used in query definition of a view?

* Q. What happens when you a delete a tuple from `player` relation with foreign key constraint? 